# 01 — Data Understanding

**Goal:** Load the IBM Telco Customer Churn dataset, inspect its structure, data types, and summary statistics, and document initial observations.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.data_preprocessing import load_raw_data, validate_dataset, TARGET_COL, ID_COL

## 1.1 Load the Dataset

In [ ]:
df = load_raw_data()
print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):\n{list(df.columns)}")

## 1.2 Inspect Sample Rows

In [ ]:
df.head(10)

In [ ]:
df.tail(5)

## 1.3 Data Types and Info

In [ ]:
df.info()

In [ ]:
df.dtypes.value_counts()

## 1.4 Summary Statistics

In [ ]:
# Numeric columns
df.describe()

In [ ]:
# Categorical columns
df.describe(include='object')

## 1.5 Target Variable Inspection

In [ ]:
print("Target column:", TARGET_COL)
print("\nValue counts:")
print(df[TARGET_COL].value_counts())
print(f"\nChurn rate: {(df[TARGET_COL] == 'Yes').mean():.4f} ({(df[TARGET_COL] == 'Yes').mean()*100:.1f}%)")

## 1.6 Dataset Validation Report

In [ ]:
report = validate_dataset(df)
for key, value in report.items():
    if key != 'dtypes':
        print(f"{key}: {value}")

## 1.7 Missing Values

In [ ]:
missing = df.isnull().sum()
print("Columns with missing values:")
print(missing[missing > 0] if missing.any() else "None")

# Check for blank strings (common in TotalCharges)
for col in df.select_dtypes(include='object').columns:
    blanks = (df[col].str.strip() == '').sum()
    if blanks > 0:
        print(f"\n'{col}' has {blanks} blank string values (not NaN)")

## 1.8 Unique Values per Column

In [ ]:
for col in df.columns:
    n_unique = df[col].nunique()
    if n_unique <= 10:
        print(f"{col} ({n_unique} unique): {df[col].unique().tolist()}")
    else:
        print(f"{col} ({n_unique} unique): [showing first 5] {df[col].unique()[:5].tolist()}")

## 1.9 Potential Leakage Check

Fields to exclude from modelling:
- `customerID`: Unique identifier — no predictive value.
- `Churn`: Target variable — must never be used as a feature.

Fields to investigate:
- `TotalCharges`: Correlated with tenure, but represents historical data available at prediction time. **Acceptable** to include.

## 1.10 Key Observations

- The dataset has ~7,043 rows and 21 columns.
- Target variable `Churn` has two classes (Yes/No) with an approximate 26.5% churn rate — moderate class imbalance.
- `TotalCharges` is stored as a string (object type) and contains blank values that need conversion to numeric NaN.
- `SeniorCitizen` is encoded as 0/1 integers unlike other binary categorical columns.
- No obvious leakage concerns beyond the identifiers noted above.
- Most categorical features have 2-4 unique values.